## Wykład 2 - LLMy
Wprowadzenie do biblioteki PyTorch i operacji wykorzystywanych w algorytmach klasyfikacji (softmax, entropia krzyżowa) oraz wskazówki do lab 1

PyTorch zapewnia:
* Obliczenia numeryczne na wielowymiarowych macierzach (tensorach)
* Automatyczne wyznaczanie gradientów (potrzebne do optymalizacji gradientowej)
* Wysokopoziomowe moduły do budowy sieci neuronowych
* Zapewnia większą elastyczność niż Keras i jest dużo przyjemniejszy w obsłudze niż Tensorflow 

In [1]:
import torch

### Macierze PyTorch

In [2]:
torch.zeros(4, 4)

tensor([[0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.]])

In [3]:
A = torch.arange(1, 10)
A

tensor([1, 2, 3, 4, 5, 6, 7, 8, 9])

In [4]:
# shape - sprawdza kształt tensora
A.shape

torch.Size([9])

In [5]:
# reshape zmienia kształt
A = A.reshape(3, 3)
A

tensor([[1, 2, 3],
        [4, 5, 6],
        [7, 8, 9]])

In [6]:
A.shape

torch.Size([3, 3])

In [7]:
# podstawienie wartości
A[0, 0] = 100

In [8]:
A

tensor([[100,   2,   3],
        [  4,   5,   6],
        [  7,   8,   9]])

In [9]:
# wybór całego wiersza (wiersz o indeksie 1)
A[1]

tensor([4, 5, 6])

In [10]:
A.sum(axis=0) # sumy w kolumnach

tensor([111,  15,  18])

In [11]:
A.sum(axis=1) # sumy w wierszach

tensor([105,  15,  24])

In [12]:
# keepdims=True zachowuje liczbę wymiarów tensora po operacji sumowania
A.sum(axis=1, keepdims=True)

tensor([[105],
        [ 15],
        [ 24]])

In [13]:
A + 3

tensor([[103,   5,   6],
        [  7,   8,   9],
        [ 10,  11,  12]])

In [14]:
A * 3

tensor([[300,   6,   9],
        [ 12,  15,  18],
        [ 21,  24,  27]])

In [15]:
# mnożenie macierzy
A @ A

tensor([[10029,   234,   339],
        [  462,    81,    96],
        [  795,   126,   150]])

#### Konkatenacja macierzy

In [16]:
a = torch.arange(1, 5)
b = torch.arange(1, 5)
print(a)
print(b)

tensor([1, 2, 3, 4])
tensor([1, 2, 3, 4])


In [17]:
torch.stack([a, b])

tensor([[1, 2, 3, 4],
        [1, 2, 3, 4]])

#### Losowanie

In [18]:
torch.randint(high=100, size=(2, 2))

tensor([[10,  6],
        [32, 20]])

In [19]:
torch.normal(mean=0, std=0.01, size=(2, 2))

tensor([[0.0067, 0.0104],
        [0.0113, 0.0032]])

##### Losowanie z zadanym rozkładem prawdopodobieństwa (numpy)
* `np.random.choice(n, p=p)` losuje liczbę ze zbioru $0, 1, ..., n-1$
* `p` - prawdopodobieństwa kolejnych wartości (muszą sumować się do 1)
* `np.random.seed` ustawia ziarno generatora - kolejne losowania dają wtedy powtarzalne wyniki

In [20]:
import numpy as np

In [21]:
p = np.array([0.1, 0.2, 0.7])
np.random.choice(3, p=p) # można uruchomić kilka razy - najczęściej wylosuje się 2

2

In [22]:
# size - liczba losowań
np.random.choice(3, size=20, p=p)

array([2, 2, 1, 1, 2, 2, 2, 2, 2, 0, 2, 0, 2, 2, 2, 2, 2, 2, 2, 0])

In [23]:
np.random.seed(1)
print(np.random.choice(3, size=10, p=p))
np.random.seed(1)
print(np.random.choice(3, size=10, p=p)) # ten sam wynik

[2 2 0 2 1 0 1 2 2 2]
[2 2 0 2 1 0 1 2 2 2]


#### Logity
- dla sieci neuronowej klasyfikującej $n$ kategorii mamy w warstwie wyjściowej $n$ neuronów (np. klasyfikacja 200k tokenów)
- **Logity** to surowe wartości przed obliczeniem funkcji aktywacji
- Rozważmy przykład:
    - dwóch przykładów (wejść sieci) o czterech cechach ($x$)
    - wektora wag $W$ o wymiarze 4 x 3
    - klasyfikacji do 3 klas (czyli 3 neurony wyjściowe) 

In [24]:
x = torch.normal(mean=0, std=1, size=(2, 4))
x

tensor([[ 2.1504,  0.6923,  0.5905,  0.8705],
        [-0.8287,  0.3562,  1.0504,  0.2100]])

In [25]:
W = torch.normal(mean=0, std=1, size=(4, 3))
W

tensor([[ 0.0647, -0.0236, -1.2597],
        [ 0.6080, -0.5607, -2.0582],
        [-0.4920,  0.1243,  1.1012],
        [-2.0210, -0.3188, -1.0513]])

In [26]:
logits = x @ W
print(logits.shape)
print(logits)

torch.Size([2, 3])
tensor([[-1.4899, -0.6431, -4.3986],
        [-0.7782, -0.1165,  1.2467]])


#### Softmax
$$softmax(z_i) = \frac{e^{z_i}}{\sum_{j=1}^{n} e^{z_j}}$$


In [27]:
exp_logits = torch.exp(logits)
exp_logits

tensor([[0.2254, 0.5257, 0.0123],
        [0.4592, 0.8900, 3.4790]])

In [28]:
exp_logits.sum(axis=1, keepdims=True)

tensor([[0.7634],
        [4.8283]])

In [29]:
# normalizujemy każdy wiersz, żeby sumował się do jedynki
# wyjścia po funkcji aktywacji softmax mają interpretację prawdopodobieństwa
probs = exp_logits/exp_logits.sum(axis=1, keepdims=True)
probs

tensor([[0.2953, 0.6886, 0.0161],
        [0.0951, 0.1843, 0.7205]])

In [30]:
# po co jest keepdims? - wymiary muszą się zgadzać
probs = exp_logits/exp_logits.sum(axis=1)
probs

RuntimeError: The size of tensor a (3) must match the size of tensor b (2) at non-singleton dimension 1

In [31]:
# prawdopodobieństwa klas dla pierwszego przykładu (pierwszy wiersz)
probs[0]

tensor([0.2953, 0.6886, 0.0161])

In [32]:
# losujemy klasę zgodnie z wyznaczonymi prawdopodobieństwami
# np.random.choice wymaga tablicy numpy - detach() odłącza tensor od obliczeń gradientów, numpy() zamienia go na tablicę numpy
np.random.choice(3, p=probs[0].detach().numpy())

1

#### One hot encoding

In [33]:
import torch.nn.functional as F

In [34]:
# pierwszy argument - jaką liczbę zakodować
# drugi argument - ile jest klas (num_classes)
# kodowanie umieszcza jedynkę na pozycji odpowiadającej indeksowi pierwszego argumentu, w wektorze o długości 
# n_classes
F.one_hot(torch.tensor(3), num_classes=5)

tensor([0, 0, 0, 1, 0])

In [35]:
# prawidłowe wyjścia sieci (etykiety) zakodowane jako one hot encoding
# dla pierwszego przykładu prawidłową etykietą jest 0, a dla drugiego 2
y = F.one_hot(torch.tensor([0, 2]), num_classes=3)
y

tensor([[1, 0, 0],
        [0, 0, 1]])

#### Entropia krzyżowa
(dla pojedynczego przykładu)


$loss = -\sum_{i=1}^{n} y_i \log(\hat{y}_i)$ 

In [36]:
probs

tensor([[0.2953, 0.6886, 0.0161],
        [0.0951, 0.1843, 0.7205]])

In [37]:
torch.log(probs)

tensor([[-1.2199, -0.3730, -4.1286],
        [-2.3527, -1.6910, -0.3277]])

In [38]:
# mnożenie (element po elemencie) wybiera logproby dla prawidłowych etykiet
y * torch.log(probs)

tensor([[-1.2199, -0.0000, -0.0000],
        [-0.0000, -0.0000, -0.3277]])

In [39]:
# sumowanie wszystkich przykładów - uwaga nawias jest potrzebny
(y * torch.log(probs)).sum()

tensor(-1.5477)

In [40]:
# pełna entropia krzyżowa - dodajemy minus i dzielimy przez liczbę przykładów, aby mieć średni koszt dla przykładu
- (y * torch.log(probs)).sum()/2

tensor(0.7738)

#### Gradienty
Źródło: [dokumentacja PyTorch](https://pytorch.org/tutorials/beginner/blitz/autograd_tutorial.html)

$$ Q = 3a^3 - b^2$$
$$\frac{\partial Q}{\partial a} = 9a^2$$
$$\frac{\partial Q}{\partial b} = -2b$$

In [41]:
a = torch.tensor([2., 3.], requires_grad=True)
b = torch.tensor([6., 4.], requires_grad=True)
Q = (3*a**3 - b**2).sum()
print(Q)

tensor(53., grad_fn=<SumBackward0>)


In [42]:
# wywołanie funkcji backward wylicza gradienty i zapisuje je w atrybucie grad odpowiednich tensorów
Q.backward()

In [43]:
a.grad

tensor([36., 81.])

In [44]:
b.grad

tensor([-12.,  -8.])

#### Prosta pętla optymalizacji gradientowej
Chcemy znaleźć minimum funkcji:
$$f(x)=(x−3)^2$$

In [45]:
x = torch.tensor([0.0], requires_grad=True)  # zaczynamy z x = 0

learning_rate = 0.1 # współczynnik uczenia
epochs = 30  # liczba iteracji

for epoch in range(epochs):
    loss = (x - 3) ** 2  # wartość funkcji kosztu
    
    loss.backward()  # wyznaczamy gradient
    
    with torch.no_grad():  # aktualizujemy wartość x - nie chcemy, żeby ta operacja była uwzględniana w wyliczaniu gradientów
        x -= learning_rate * x.grad
        x.grad.zero_()  # Zerujemy gradienty przed kolejną pętlą - to ważne!

    print(f"Krok {epoch+1}: x = {x.item()}, Loss = {loss.item()}")

print(f"Końcowa wartość x: {x.item()}")


Krok 1: x = 0.6000000238418579, Loss = 9.0
Krok 2: x = 1.0800000429153442, Loss = 5.760000228881836
Krok 3: x = 1.4639999866485596, Loss = 3.6863999366760254
Krok 4: x = 1.7711999416351318, Loss = 2.3592960834503174
Krok 5: x = 2.0169599056243896, Loss = 1.5099495649337769
Krok 6: x = 2.2135679721832275, Loss = 0.9663678407669067
Krok 7: x = 2.370854377746582, Loss = 0.6184753179550171
Krok 8: x = 2.4966835975646973, Loss = 0.39582422375679016
Krok 9: x = 2.597346782684326, Loss = 0.2533273994922638
Krok 10: x = 2.677877426147461, Loss = 0.16212961077690125
Krok 11: x = 2.7423019409179688, Loss = 0.10376295447349548
Krok 12: x = 2.793841600418091, Loss = 0.06640829145908356
Krok 13: x = 2.835073232650757, Loss = 0.042501285672187805
Krok 14: x = 2.868058681488037, Loss = 0.027200838550925255
Krok 15: x = 2.894446849822998, Loss = 0.01740851067006588
Krok 16: x = 2.915557384490967, Loss = 0.011141467839479446
Krok 17: x = 2.932446002960205, Loss = 0.007130555342882872
Krok 18: x = 2.945

In [46]:
x

tensor([2.9963], requires_grad=True)

In [47]:
# metoda item wyjmuje liczbę z tensora (tylko dla jednowymiarowych)
x.item()

2.996286153793335

### Python wskazówki do lab

In [48]:
text = 'ndckdcnoeicmo'

In [49]:
# zbiór (set) zawiera tylko unikalne elementy
chars = set(text)
chars

{'c', 'd', 'e', 'i', 'k', 'm', 'n', 'o'}

In [50]:
#zamiana na listę
chars = list(chars)
chars

['o', 'm', 'k', 'e', 'n', 'd', 'c', 'i']

In [51]:
# długość listy
len(chars)

8

In [52]:
# sortowanie
sorted(chars)

['c', 'd', 'e', 'i', 'k', 'm', 'n', 'o']

In [53]:
for i, c in enumerate(chars):
    print(i, c)

0 o
1 m
2 k
3 e
4 n
5 d
6 c
7 i


#### Słowniki

In [54]:
d = {0: 'a', 1: 'b', 2: 'c'}

In [55]:
d[0]

'a'

### List comprehensions

In [56]:
[i for i in range(10)]

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

In [57]:
[i**2 for i in range(10)]

[0, 1, 4, 9, 16, 25, 36, 49, 64, 81]

In [58]:
[d[i] for i in [1, 0, 1, 2, 2, 1]]

['b', 'a', 'b', 'c', 'c', 'b']

In [59]:
list(enumerate(chars))

[(0, 'o'),
 (1, 'm'),
 (2, 'k'),
 (3, 'e'),
 (4, 'n'),
 (5, 'd'),
 (6, 'c'),
 (7, 'i')]

In [60]:
{i: c for i, c in enumerate(chars)}

{0: 'o', 1: 'm', 2: 'k', 3: 'e', 4: 'n', 5: 'd', 6: 'c', 7: 'i'}

In [61]:
{c: i for i, c in enumerate(chars)}

{'o': 0, 'm': 1, 'k': 2, 'e': 3, 'n': 4, 'd': 5, 'c': 6, 'i': 7}

##### Zip
* łączy elementy iterowalne (np. listy) tworząc pary (lub krotki) z różnych list
* Kończy jak krótszy z elementów iterowalnych się wyczerpie

In [62]:
lista1 = ['a', 'b', 'c']
lista2 = ['d', 'e', 'f']

In [63]:
print(list(zip(lista1, lista2)))

[('a', 'd'), ('b', 'e'), ('c', 'f')]


In [64]:
for ch1, ch2 in zip(lista1, lista2):
    print(ch1, ch2)

a d
b e
c f


In [65]:
# teraz bierzemy z drugiej listy od elementu o indeksie 1 i mamy tylko dwie pary
for ch1, ch2 in zip(lista1, lista2[1:]):
    print(ch1, ch2)

a e
b f
